# Long Eddy Case Study

Select strongly propagating long-lived AE/CE examples and compare tilt with surface and bottom motion.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import seacofs_tilt_tools as tilt

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies = pd.read_pickle(paths.beta_eddies)
dic_vert = tilt.load_vertical_dictionary(paths)
df_eddies = tilt.add_top_bottom_speeds(df_eddies, dic_vert)

prop_dist = df_eddies.groupby("Eddy")[["xc", "yc"]].apply(lambda g: np.nansum(np.hypot(g.xc.diff(), g.yc.diff())))
df_eddies["PropDist"] = df_eddies["Eddy"].map(prop_dist)
selected = {cyc: df_eddies.loc[df_eddies.Cyc == cyc].groupby("Eddy")["lat"].agg(np.ptp).idxmax() for cyc in ["AE", "CE"]}
selected


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 6), constrained_layout=True)
for row, (cyc, eddy_id) in enumerate(selected.items()):
    df = df_eddies.loc[df_eddies.Eddy == eddy_id].copy().sort_values("Day")
    color = "darkred" if cyc == "AE" else "navy"

    ax = axs[row, 0]
    ax.plot(df.xc, df.yc, color=color, lw=1.8)
    sc = ax.scatter(df.xc, df.yc, c=df.Day, s=18, cmap="viridis", zorder=3)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(f"{cyc}{int(eddy_id)} track")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")

    ax = axs[row, 1]
    ax.plot(df.Day, df.TiltDis, color=color, label="Tilt distance")
    ax.plot(df.Day, df.EddyProp, color="0.35", label="Surface prop.")
    ax.plot(df.Day, df.btm_prop, color="0.65", label="Bottom prop.")
    ax.set_title(f"{cyc}{int(eddy_id)} evolution")
    ax.set_xlabel("Day")
    ax.set_ylabel("km or m s$^{-1}$")
    ax.legend(frameon=False)
fig.colorbar(sc, ax=axs[:, 0], label="Day", shrink=0.8)
plt.show()


In [ ]:
case_summary = []
for cyc, eddy_id in selected.items():
    df = df_eddies.loc[df_eddies.Eddy == eddy_id]
    case_summary.append({
        "cyc": cyc,
        "eddy": int(eddy_id),
        "n_days": len(df),
        "tilt_median_km": df.TiltDis.median(),
        "tilt_iqr_km": df.TiltDis.quantile(0.75) - df.TiltDis.quantile(0.25),
        "prop_distance_km": df.PropDist.iloc[0],
    })
pd.DataFrame(case_summary)
